# SVM (Support Vector Machine)
## algoritma supervised learning untuk klasifikasi dan regresi
### menemukan hyperplane optimal sebagai bidang pemisah terbaik untuk memisahkan data dari dua kelas atau lebih

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
dataset = pd.read_csv('iris-flower.csv')
dataset

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa
...,...,...,...,...,...,...
145,146,6.7,3.0,5.2,2.3,Iris-virginica
146,147,6.3,2.5,5.0,1.9,Iris-virginica
147,148,6.5,3.0,5.2,2.0,Iris-virginica
148,149,6.2,3.4,5.4,2.3,Iris-virginica


In [3]:
dataset['Species'].unique()

array(['Iris-setosa', 'Iris-versicolor', 'Iris-virginica'], dtype=object)

In [4]:
le = LabelEncoder()
dataset['Species_encoded'] = le.fit_transform(dataset['Species'])

x = dataset.drop(columns=['Id', 'Species', 'Species_encoded'], errors = 'ignore')
y = dataset['Species_encoded']
le

LabelEncoder()

In [5]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.3, random_state=0)

In [6]:
pipeline = Pipeline([('scaler', StandardScaler()),('svc',SVC())])

In [7]:
params = {
    'svc__kernel' : ['poly', 'rbf', 'sigmoid'],
    'svc__C' : [0.5, 1, 10, 100],
    'svc__gamma' : ['scale', 1, 0.1, 0.001]
}

In [8]:
grid_search = GridSearchCV(
    estimator = pipeline,
    param_grid = params,
    scoring = 'accuracy',
    cv = 5,
    n_jobs = 1,
    verbose = 1
)

In [9]:
print(type(x), x_train.shape)
print(type(y), y_train.shape)

grid_search.fit(x_train, y_train)
print ("Grid Search Done!")

<class 'pandas.core.frame.DataFrame'> (105, 4)
<class 'pandas.core.series.Series'> (105,)
Fitting 5 folds for each of 48 candidates, totalling 240 fits
Grid Search Done!


In [10]:
print("Best Params: ", grid_search.best_params_)

Best Params:  {'svc__C': 1, 'svc__gamma': 'scale', 'svc__kernel': 'rbf'}


In [11]:
best_model = grid_search.best_estimator_

In [12]:
y_train_pred = best_model.predict(x_train)
y_test_pred = best_model.predict(x_test)

print("Train Accuracy: ", accuracy_score(y_train, y_train_pred))
print("Test Accuracy: ", accuracy_score(y_test, y_test_pred))

Train Accuracy:  0.9714285714285714
Test Accuracy:  0.9777777777777777


In [13]:
print("Classification Report (test): \n",
      classification_report(y_test, y_test_pred, target_names=le.classes_))
print("Confusion Matrix (test: \n", confusion_matrix(y_test, y_test_pred))

Classification Report (test): 
                  precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        16
Iris-versicolor       1.00      0.94      0.97        18
 Iris-virginica       0.92      1.00      0.96        11

       accuracy                           0.98        45
      macro avg       0.97      0.98      0.98        45
   weighted avg       0.98      0.98      0.98        45

Confusion Matrix (test: 
 [[16  0  0]
 [ 0 17  1]
 [ 0  0 11]]


In [15]:
iris_anonim = pd.DataFrame([[6.5, 3.0, 7.0, 4.8]], columns=x.columns)
pred = best_model.predict(iris_anonim)
print("Predicted label (encoded): ", pred)
print("Predicted Species : ", le.inverse_transform(pred))

Predicted label (encoded):  [2]
Predicted Species :  ['Iris-virginica']
